<a href="https://colab.research.google.com/github/anupbagale/python-for-ai-ml/blob/main/16-time-series-and-recommendations/03_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 16: Solutions

Try the exercises first. The notes after each solution explain the idea behind it and the mistake each check is designed to catch.

In [1]:
# Setup: run this cell once before you start.
# It defines the check functions used below.

import math
import warnings

import numpy as np
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import TimeSeriesSplit


class Raises:
    """Marks a test case where the code is expected to raise an error."""
    def __init__(self, exc):
        self.exc = exc

    def __repr__(self):
        return f"{self.exc.__name__} to be raised"


def _same(a, b, tol=1e-6):
    if isinstance(a, (pd.DataFrame, pd.Series)) or isinstance(b, (pd.DataFrame, pd.Series)):
        if type(a) is not type(b):
            return False
        options = dict(check_dtype=False, check_exact=False, rtol=tol, atol=tol, check_index_type=False)
        try:
            if isinstance(a, pd.DataFrame):
                pd.testing.assert_frame_equal(a, b, check_column_type=False, **options)
            else:
                pd.testing.assert_series_equal(a, b, **options)
            return True
        except AssertionError:
            return False
    if isinstance(a, np.ndarray) or isinstance(b, np.ndarray):
        try:
            a_arr, b_arr = np.asarray(a), np.asarray(b)
        except Exception:
            return False
        if a_arr.shape != b_arr.shape:
            return False
        if a_arr.dtype.kind in "fc" or b_arr.dtype.kind in "fc":
            return bool(np.allclose(a_arr, b_arr, rtol=tol, atol=tol, equal_nan=True))
        return bool(np.array_equal(a_arr, b_arr))
    if isinstance(a, (float, np.floating)) or isinstance(b, (float, np.floating)):
        if not isinstance(a, (int, float, np.number)) or not isinstance(b, (int, float, np.number)):
            return False
        if math.isnan(a) or math.isnan(b):
            return False
        return math.isclose(a, b, rel_tol=tol, abs_tol=tol)
    if isinstance(a, (list, tuple)) and isinstance(b, (list, tuple)):
        return len(a) == len(b) and all(_same(x, y) for x, y in zip(a, b))
    if isinstance(a, dict) and isinstance(b, dict):
        return a.keys() == b.keys() and all(_same(a[k], b[k]) for k in a)
    return a == b


def _short(x, limit=90):
    if isinstance(x, (pd.DataFrame, pd.Series)):
        lines = (x.to_string(max_rows=6, max_cols=6) if isinstance(x, pd.DataFrame) else x.to_string(max_rows=6)).splitlines()
        return f"{type(x).__name__} shape={x.shape}\n          " + "\n          ".join(lines[:8])
    if isinstance(x, np.ndarray):
        body = np.array2string(x, precision=4, threshold=12, edgeitems=3, separator=", ")
        text = f"array(shape={x.shape}, dtype={x.dtype}, {' '.join(body.split())})"
    elif isinstance(x, tuple) and any(isinstance(i, np.ndarray) for i in x):
        text = "(" + ", ".join(_short(i, limit) for i in x) + ")"
        return text
    else:
        text = repr(x)
    if isinstance(x, (pd.DataFrame, pd.Series)):
        return text
    return text if len(text) <= limit else text[:limit - 3] + "..."


def _scenarios(name, cases):
    for description, scenario, expected, hint in cases:
        try:
            result = scenario()
        except Exception as e:
            if isinstance(expected, Raises) and isinstance(e, expected.exc):
                continue
            print(f"FAIL  {name}: {description}")
            print(f"      raised {type(e).__name__}: {e}")
            if isinstance(expected, Raises):
                print(f"      expected {expected!r}")
            else:
                print(f"      expected: {_short(expected)}")
            if hint:
                print(f"      hint: {hint}")
            return
        if isinstance(expected, Raises):
            print(f"FAIL  {name}: {description}")
            print(f"      expected {expected!r}, but nothing was raised")
            if hint:
                print(f"      hint: {hint}")
            return
        if not _same(result, expected):
            print(f"FAIL  {name}: {description}")
            print(f"      expected: {_short(expected)}")
            print(f"      got:      {_short(result)}")
            if hint:
                print(f"      hint: {hint}")
            return
    print(f"PASS  all {len(cases)} tests passed for {name}")


# ---- checks ----

def _series(n=60, seed=0):
    rng = np.random.default_rng(seed)
    idx = pd.date_range("2024-01-01", periods=n, freq="D")
    return pd.Series(100 + np.arange(n) + 10 * np.sin(np.arange(n) * 2 * np.pi / 7) + rng.normal(0, 3, n), index=idx)


def check_lag_features(f):
    s = _series()
    expected = pd.DataFrame({"lag_1": s.shift(1), "lag_7": s.shift(7), "lag_14": s.shift(14)}, index=s.index)
    _scenarios("lag_features", [
        ("the example", lambda: f(pd.Series([10.0, 20, 30, 40]), [1, 2]),
         pd.DataFrame({"lag_1": [np.nan, 10, 20, 30], "lag_2": [np.nan, np.nan, 10, 20]}), "shift(k) looks k steps into the past"),
        ("a dated series keeps its index", lambda: f(s, [1, 7, 14]), expected, None),
        ("columns in the order given", lambda: list(f(s, [7, 1]).columns), ["lag_7", "lag_1"], None),
        ("lag 0 is the series itself", lambda: f(s, [0])["lag_0"].tolist(), s.tolist(), None),
    ])


def check_rolling_features(f):
    s = _series()
    past = s.shift(1).rolling(7)
    expected = pd.DataFrame({"mean_7": past.mean(), "min_7": past.min(), "max_7": past.max()}, index=s.index)
    spike = pd.Series([1.0, 1, 1, 1, 100])
    _scenarios("rolling_features", [
        ("the example", lambda: f(pd.Series([1.0, 2, 3, 4, 5]), 2)["mean_2"].to_numpy(),
         np.array([np.nan, np.nan, 1.5, 2.5, 3.5]), "shift(1) before rolling"),
        ("mean, min and max of the previous 7 days", lambda: f(s, 7), expected, None),
        ("the current value is never included", lambda: float(f(spike, 3)["max_3"].iloc[-1]), 1.0,
         "the window must end one step before t"),
        ("NaN until a full window exists", lambda: int(f(s, 7)["mean_7"].isna().sum()), 7, None),
    ])


def check_seasonal_naive(f):
    _scenarios("seasonal_naive", [
        ("the example", lambda: f([1, 2, 3, 10, 20, 30], 3, 5), np.array([10.0, 20, 30, 10, 20]), "repeat the last full season"),
        ("season 1 is the naive forecast", lambda: f([5, 6, 7], 1, 4), np.array([7.0, 7, 7, 7]), None),
        ("a horizon shorter than a season", lambda: f(np.arange(14), 7, 3), np.array([7.0, 8, 9]), None),
        ("returns floats", lambda: f([1, 2], 1, 2).dtype.kind, "f", None),
        ("too little history raises", lambda: f([1, 2], 7, 3), Raises(ValueError), None),
    ])


def check_precision_recall_at_k(f):
    _scenarios("precision_recall_at_k", [
        ("a small example", lambda: f(["a", "b", "c", "d"], {"b", "d", "x"}, 3), (1 / 3, 1 / 3),
         "only the first k recommendations count"),
        ("all hits", lambda: f([1, 2], [1, 2], 2), (1.0, 1.0), None),
        ("a short list is penalised", lambda: f([1], [1, 2], 5), (0.2, 0.5), "divide by k, not by the list length"),
        ("relevant can be any collection", lambda: f(np.array([3, 4, 5]), np.array([5, 9]), 3), (1 / 3, 0.5), None),
        ("returns floats", lambda: [type(v).__name__ for v in f([1], [1], 1)], ["float", "float"], None),
        ("no relevant items raises", lambda: f([1, 2], [], 2), Raises(ValueError), None),
    ])


def check_forecast_errors(f):
    rng = np.random.default_rng(1)
    train = 50 + 10 * np.sin(np.arange(140) * 2 * np.pi / 7) + rng.normal(0, 2, 140)
    actual = 50 + 10 * np.sin(np.arange(140, 168) * 2 * np.pi / 7) + rng.normal(0, 2, 28)
    forecast = np.resize(train[-7:], 28)
    e = actual - forecast
    _scenarios("forecast_errors", [
        ("a small example", lambda: f([3, 5], [1, 5], [1, 2, 4, 7]), {"MAE": 1.0, "RMSE": math.sqrt(2), "MASE": 0.5},
         "naive training errors: |2-1|, |4-2|, |7-4|, mean 2"),
        ("a weekly series with season_length 7", lambda: f(actual, forecast, train, 7),
         {"MAE": float(np.mean(np.abs(e))), "RMSE": float(np.sqrt(np.mean(e ** 2))),
          "MASE": float(np.mean(np.abs(e)) / np.mean(np.abs(train[7:] - train[:-7])))}, "compare train[t] with train[t - season_length]"),
        ("the right season beats the wrong one", lambda: f(actual, forecast, train, 7)["MASE"] > f(actual, forecast, train, 1)["MASE"], True,
         "a weekly series is predicted much better by last week than by yesterday, so its scale is smaller"),
        ("different lengths raise", lambda: f([1, 2, 3], [1, 2], [1, 2, 3]), Raises(ValueError), None),
        ("one forecast for three actual values raises", lambda: f([1, 2, 3], [2], [1, 2, 3]), Raises(ValueError),
         "NumPy would broadcast a single forecast, so compare the lengths yourself"),
        ("a constant training series raises", lambda: f([1, 2], [1, 2], [5, 5, 5]), Raises(ValueError), None),
    ])


def check_expanding_window_splits(f):
    def ref(n, n_splits, test_size, gap=0):
        return [(tr, te) for tr, te in TimeSeriesSplit(n_splits=n_splits, test_size=test_size, gap=gap).split(np.zeros(n))]

    def as_lists(splits):
        return [(np.asarray(a).tolist(), np.asarray(b).tolist()) for a, b in splits]

    def all_before():
        return all(np.max(tr) < np.min(te) for tr, te in f(500, 6, 30, gap=5))

    _scenarios("expanding_window_splits", [
        ("a small example", lambda: as_lists(f(10, 3, 2)), [(list(range(0, 4)), [4, 5]), (list(range(0, 6)), [6, 7]), (list(range(0, 8)), [8, 9])],
         "test blocks cover the end of the series"),
        ("matches TimeSeriesSplit", lambda: as_lists(f(731, 5, 60)), as_lists(ref(731, 5, 60)), None),
        ("with a gap", lambda: as_lists(f(731, 4, 30, gap=7)), as_lists(ref(731, 4, 30, 7)), "leave out gap indices before each test block"),
        ("training always comes before testing", all_before, True, None),
        ("too many splits raise", lambda: f(10, 5, 2), Raises(ValueError), None),
    ])


def check_exponential_smoothing(f):
    s = _series(80, seed=3)
    ref_level = s.ewm(alpha=0.3, adjust=False).mean()
    _scenarios("exponential_smoothing", [
        ("a small example", lambda: f([10, 20, 30], 0.5), (np.array([np.nan, 10.0, 15.0]), 22.5),
         "forecast[t] is the level before seeing y[t]"),
        ("matches pandas ewm", lambda: f(s, 0.3), (ref_level.shift(1).to_numpy(), float(ref_level.iloc[-1])), None),
        ("alpha 1 is the naive forecast", lambda: f([3, 8, 1, 4], 1.0), (np.array([np.nan, 3.0, 8.0, 1.0]), 4.0), None),
        ("alpha 0 raises", lambda: f([1, 2], 0), Raises(ValueError), None),
    ])


def check_recursive_forecast(f):
    class OnlyLag1:
        def predict(self, X):
            X = np.asarray(X)
            assert X.ndim == 2 and X.shape[0] == 1, "predict should get a 2D array with one row"
            return X[:, 0] * 2

    class Average:
        def predict(self, X):
            return np.asarray(X).mean(axis=1)

    rng = np.random.default_rng(4)
    y = np.zeros(300)
    for t in range(2, 300):
        y[t] = 0.6 * y[t - 1] - 0.3 * y[t - 2] + rng.normal()
    lags = np.column_stack([y[1:-1], y[:-2]])
    ar = LinearRegression().fit(lags, y[2:])

    def manual_ar(history, horizon):
        h = list(history)
        out = []
        for _ in range(horizon):
            p = ar.intercept_ + ar.coef_[0] * h[-1] + ar.coef_[1] * h[-2]
            out.append(p)
            h.append(p)
        return np.array(out)

    _scenarios("recursive_forecast", [
        ("the most recent value comes first", lambda: f(OnlyLag1(), [1, 2, 3], 3, 3), np.array([6.0, 12.0, 24.0]),
         "row = [y[t-1], y[t-2], ...]: reverse the last n_lags values"),
        ("forecasts are fed back in", lambda: f(Average(), [0, 0, 6], 3, 3), np.array([2.0, 8 / 3, 32 / 9]), None),
        ("matches a fitted autoregressive model", lambda: f(ar, y[:250], 2, 10), manual_ar(y[:250], 10), None),
        ("too little history raises", lambda: f(Average(), [1], 3, 2), Raises(ValueError), None),
    ])


def check_item_cosine_similarity(f):
    rng = np.random.default_rng(5)
    R = (rng.random((50, 12)) < 0.2).astype(float)
    R[:, 3] = 0                                         # an item nobody bought
    ref = cosine_similarity(R.T)
    np.fill_diagonal(ref, 0)
    ratings = rng.integers(0, 6, (40, 8)).astype(float)
    ref2 = cosine_similarity(ratings.T)
    np.fill_diagonal(ref2, 0)
    _scenarios("item_cosine_similarity", [
        ("a small example", lambda: f(np.array([[1, 1, 0], [1, 0, 1], [0, 0, 1]])),
         np.array([[0, 1 / math.sqrt(2), 0.5], [1 / math.sqrt(2), 0, 0], [0.5, 0, 0]]), "dot products divided by the column lengths"),
        ("matches scikit-learn on purchases, with an empty item", lambda: f(R), ref, "an item with no entries gets similarity 0"),
        ("matches scikit-learn on ratings", lambda: f(ratings), ref2, None),
        ("the diagonal is 0", lambda: float(np.abs(np.diag(f(ratings))).max()), 0.0, None),
    ])


def check_holt_forecast(f):
    rng = np.random.default_rng(6)
    y = np.cumsum(rng.normal(1, 2, 60)) + 50

    def ref(series, a, b, h):
        level, trend = series[0], series[1] - series[0]
        for t in range(1, len(series)):
            new = a * series[t] + (1 - a) * (level + trend)
            trend = b * (new - level) + (1 - b) * trend
            level = new
        return level + trend * np.arange(1, h + 1)

    bend = np.r_[np.arange(10.0), 10 + 3 * np.arange(1, 11)]
    _scenarios("holt_forecast", [
        ("a straight line continues exactly", lambda: f(np.arange(5, 25, 2.0), 0.3, 0.2, 3), np.array([25.0, 27.0, 29.0]), None),
        ("matches the reference", lambda: f(y, 0.5, 0.3, 6), ref(y, 0.5, 0.3, 6), "update the trend with the change from the old level to the new one"),
        ("a series whose trend changes", lambda: f(bend, 0.8, 0.5, 4), ref(bend, 0.8, 0.5, 4), None),
        ("forecasts can go beyond the history", lambda: bool(f(np.arange(10.0), 0.5, 0.5, 5).max() > 9), True, None),
        ("one value raises", lambda: f([3.0], 0.5, 0.5, 2), Raises(ValueError), None),
        ("beta of 0 raises", lambda: f([1.0, 2.0], 0.5, 0, 2), Raises(ValueError), None),
    ])


def check_als_factorise(f):
    rng = np.random.default_rng(7)
    n_u, n_i, k = 40, 30, 2
    P_true, Q_true = rng.normal(size=(n_u, k)), rng.normal(size=(n_i, k))
    mask = rng.random((n_u, n_i)) < 0.4
    mask[5] = False                                       # a user with no entries
    users, items = np.nonzero(mask)
    values = np.sum(P_true[users] * Q_true[items], axis=1) + rng.normal(0, 0.1, len(users))

    def ref(n_factors, reg, n_iter, seed=0):
        g = np.random.default_rng(seed)
        P = g.normal(0, 0.1, size=(n_u, n_factors))
        Q = g.normal(0, 0.1, size=(n_i, n_factors))
        for _ in range(n_iter):
            for u in range(n_u):
                r = np.flatnonzero(users == u)
                P[u] = 0.0 if len(r) == 0 else np.linalg.solve(Q[items[r]].T @ Q[items[r]] + reg * np.eye(n_factors), Q[items[r]].T @ values[r])
            for i in range(n_i):
                r = np.flatnonzero(items == i)
                Q[i] = 0.0 if len(r) == 0 else np.linalg.solve(P[users[r]].T @ P[users[r]] + reg * np.eye(n_factors), P[users[r]].T @ values[r])
        return P, Q

    def objective(P, Q, reg):
        return float(np.sum((values - np.sum(P[users] * Q[items], axis=1)) ** 2) + reg * (np.sum(P ** 2) + np.sum(Q ** 2)))

    def fits_well():
        P, Q = f(users, items, values, n_u, n_i, 2, 1.0, 20)
        return bool(np.sqrt(np.mean((values - np.sum(P[users] * Q[items], axis=1)) ** 2)) < 0.2)

    def never_increases():
        objs = [objective(*f(users, items, values, n_u, n_i, 2, 0.5, it), 0.5) for it in range(1, 7)]
        return bool(all(b <= a + 1e-9 for a, b in zip(objs, objs[1:])))

    _scenarios("als_factorise", [
        ("matches the reference procedure", lambda: f(users, items, values, n_u, n_i, 2, 0.5, 5), ref(2, 0.5, 5),
         "one rng: P first, then Q; users in order, then items"),
        ("another setting", lambda: f(users, items, values, n_u, n_i, 3, 2.0, 3, seed=4), ref(3, 2.0, 3, seed=4), None),
        ("a user with no entries gets zeros", lambda: f(users, items, values, n_u, n_i, 2, 0.5, 2)[0][5].tolist(), [0.0, 0.0], None),
        ("the regularised error never increases", never_increases, True, None),
        ("fits the known entries well", fits_well, True, None),
    ])


def check_recommend_top_k(f):
    scores = np.array([[0.9, 0.1, 0.5, 0.5, 0.0],
                       [0.0, 0.0, 0.0, 0.0, 0.0],
                       [1.0, 2.0, 3.0, 4.0, 5.0]])
    seen = np.array([[True, False, False, False, False],
                     [False, False, False, False, False],
                     [False, True, True, True, True]])
    popularity = np.array([10, 50, 5, 20, 30])
    rng = np.random.default_rng(8)
    S = np.round(rng.random((20, 40)), 1)
    seen_big = rng.random((20, 40)) < 0.3
    pop_big = rng.integers(0, 100, 40)

    def ref_lists():
        out = []
        for u in range(20):
            cand = [i for i in range(40) if not seen_big[u, i]]
            cand.sort(key=lambda i: (-S[u, i], -pop_big[i], i))
            out.append(cand[:7])
        return out

    _scenarios("recommend_top_k", [
        ("seen items are skipped; ties go to the more popular item", lambda: f(scores, seen, popularity, 3)[0].tolist(), [3, 2, 1],
         "items 2 and 3 tie on score; item 3 is more popular"),
        ("a user with no signal gets the most popular items", lambda: f(scores, seen, popularity, 3)[1].tolist(), [1, 4, 3], None),
        ("fewer unseen items than k: return them all", lambda: f(scores, seen, popularity, 3)[2].tolist(), [0], None),
        ("matches a reference ranking", lambda: [r.tolist() for r in f(S, seen_big, pop_big, 7)], ref_lists(), None),
        ("one array per user", lambda: len(f(S, seen_big, pop_big, 5)), 20, None),
    ])


print("Setup done. Checks are ready.")

Setup done. Checks are ready.


## Easy

### Exercise 1: `lag_features`

In [2]:
def lag_features(series: pd.Series, lags) -> pd.DataFrame:
    return pd.DataFrame({f"lag_{k}": series.shift(k) for k in lags}, index=series.index)

In [3]:
check_lag_features(lag_features)

PASS  all 4 tests passed for lag_features


`shift(k)` moves every value `k` positions later, so the value at time `t` becomes the value from time `t - k`. A dictionary comprehension builds all the columns at once, and the dictionary keeps them in the order of `lags`.

A positive shift looks into the past, which is what forecasting needs. `shift(-1)` would look one step into the **future**, and a feature built with it would be a leak. The check includes a lag of 0, which is just the series itself: legal as a column, but useless as a feature for forecasting that same value, since it *is* the value.

The rows with `NaN` at the start have to be dropped (or filled) before fitting most models. Gradient boosting in scikit-learn accepts `NaN` directly, which is convenient here.

### Exercise 2: `rolling_features`

In [4]:
def rolling_features(series: pd.Series, window: int) -> pd.DataFrame:
    past = series.shift(1).rolling(window)          # shift first, so the window ends one step before t
    return pd.DataFrame({
        f"mean_{window}": past.mean(),
        f"min_{window}": past.min(),
        f"max_{window}": past.max(),
    }, index=series.index)

In [5]:
check_rolling_features(rolling_features)

PASS  all 4 tests passed for rolling_features


The one line that matters is `shift(1)` **before** `rolling`. pandas' `rolling(window)` at time `t` covers `t - window + 1` up to and including `t`. Shifting first moves the window back by one step, so it ends at `t - 1`.

The lesson showed what happens without the shift: the "rolling mean of the last 7 days" contains the very value being forecast, and the model's error drops for a reason that won't exist when the model is used. The check confirms the window never includes the current value.

Rolling **min** and **max** are useful too: the highest demand in the last week, the lowest temperature in the last month. Rolling standard deviations measure recent volatility. All follow the same shift-then-roll rule.

### Exercise 3: `seasonal_naive`

In [6]:
def seasonal_naive(history, season_length: int, horizon: int) -> np.ndarray:
    history = np.asarray(history, dtype=float)
    if len(history) < season_length:
        raise ValueError(f"need at least {season_length} values, got {len(history)}")
    last_season = history[-season_length:]
    return np.resize(last_season, horizon)

In [7]:
check_seasonal_naive(seasonal_naive)

PASS  all 5 tests passed for seasonal_naive


`np.resize` repeats an array cyclically to the requested length, which is exactly "repeat the last season". An equivalent formula: the forecast `h` steps ahead (counting from 1) is `history[n - season_length + (h - 1) % season_length]`.

Seasonal naive is the baseline to beat whenever a series has a strong cycle: hourly electricity demand (season 24), daily shop visits (season 7), monthly sales (season 12). For the hourly bike data in the mini project, "the same hour yesterday" and "the same hour last week" are both seasonal naive forecasts, and both are surprisingly hard to beat without weather information.

### Exercise 4: `precision_recall_at_k`

In [8]:
def precision_recall_at_k(recommended, relevant, k: int):
    relevant = set(relevant)
    if not relevant:
        raise ValueError("no relevant items, so recall is undefined")
    hits = len(set(list(recommended)[:k]) & relevant)
    return float(hits / k), float(hits / len(relevant))

In [9]:
check_precision_recall_at_k(precision_recall_at_k)

PASS  all 6 tests passed for precision_recall_at_k


Turning `relevant` into a set makes membership tests fast and handles lists, arrays or sets alike. Slicing `[:k]` keeps only the items the user would see.

Precision at k and recall at k pull against each other as `k` changes: a longer list finds more relevant items (higher recall) but includes more misses (lower precision). Which matters depends on the screen: a phone shows maybe 5 items, an email digest 10, a "more like this" page 50.

To summarise over all users, average the per-user values. Users with no relevant items in the test period are usually left out (as here, by raising an error), and it's worth reporting how many there were. The **hit rate at k** (did the user get at least one relevant item?) is `hits > 0` averaged over users.

## Medium

### Exercise 5: `forecast_errors`

In [10]:
def forecast_errors(actual, forecast, train, season_length: int = 1) -> dict:
    actual, forecast, train = (np.asarray(v, dtype=float) for v in (actual, forecast, train))
    if actual.shape != forecast.shape:
        raise ValueError("actual and forecast must have the same length")
    scale = np.mean(np.abs(train[season_length:] - train[:-season_length]))
    if scale == 0:
        raise ValueError("the training series has no variation at this season length")
    errors = actual - forecast
    mae = np.mean(np.abs(errors))
    return {"MAE": float(mae), "RMSE": float(np.sqrt(np.mean(errors ** 2))), "MASE": float(mae / scale)}

In [11]:
check_forecast_errors(forecast_errors)

PASS  all 6 tests passed for forecast_errors


`train[season_length:] - train[:-season_length]` subtracts each value from the one a season later, for every position at once: these are the errors the seasonal naive forecast would have made on the training data.

Why scale by the **training** data, not the test data? So that the denominator doesn't depend on the period being forecast, and so MASE stays defined even for a single test value. Rob Hyndman and Anne Koehler proposed it in 2006 as a replacement for percentage errors, which break down when actual values are near zero (look at the MAPE values in the lesson).

The great advantage of MASE is that it's **comparable across series**: an MAE of 50 is excellent for a shop selling 10,000 items a day and terrible for one selling 60. A MASE of 0.8 means "20% better than the simple seasonal forecast", whatever the scale. That's why forecasting competitions such as the M4 used it.

### Exercise 6: `expanding_window_splits`

In [12]:
def expanding_window_splits(n_samples: int, n_splits: int, test_size: int, gap: int = 0):
    first_test = n_samples - n_splits * test_size
    if first_test - gap <= 0:
        raise ValueError("not enough samples for this many splits, this test size and this gap")
    splits = []
    for i in range(n_splits):
        test_start = first_test + i * test_size
        train = np.arange(0, test_start - gap)
        test = np.arange(test_start, test_start + test_size)
        splits.append((train, test))
    return splits

In [13]:
check_expanding_window_splits(expanding_window_splits)

PASS  all 5 tests passed for expanding_window_splits


The test blocks are laid out backwards from the end of the series, so the final block always ends at the last observation: the most recent data is always tested. Each training set grows with each split, which is why this is called an **expanding** window.

The **gap** removes the observations just before each test block from training. Why would you throw data away? If features look back several steps, or the target is a sum over the next few periods, then the last training rows and the first test rows share information, which leaks a little of the test period into training. A gap at least as long as that overlap removes it. A gap also mimics reality when the latest data arrives late: if sales figures take a week to finalise, a forecast made today can't use last week.

The check also confirms the property that matters most: in every split, **every training index comes before every test index**.

### Exercise 7: `exponential_smoothing`

In [14]:
def exponential_smoothing(y, alpha: float):
    if not 0 < alpha <= 1:
        raise ValueError(f"alpha must be in (0, 1], got {alpha}")
    y = np.asarray(y, dtype=float)
    forecasts = np.full(len(y), np.nan)
    level = y[0]
    for t in range(1, len(y)):
        forecasts[t] = level
        level = alpha * y[t] + (1 - alpha) * level
    return forecasts, float(level)

In [15]:
check_exponential_smoothing(exponential_smoothing)

PASS  all 4 tests passed for exponential_smoothing


Unrolling the update shows where the name comes from: the level is `α y_t + α(1 - α) y_{t-1} + α(1 - α)² y_{t-2} + ...`, a weighted average of all past values with weights that shrink **exponentially** with age. With `α = 0.3`, a value from 10 steps ago still has a weight of about 0.008.

`α = 1` gives the naive forecast (the level is just the last value), and as `α` approaches 0 the forecast barely moves from its starting value.

pandas has the same recursion built in: `series.ewm(alpha=alpha, adjust=False).mean()` gives the level after each observation, so `.shift(1)` of it gives these forecasts. The check uses that to confirm your version.

To choose `α`, try a range of values and keep the one with the smallest one-step error on the training period. That's what `statsmodels`' `SimpleExpSmoothing.fit()` does, with an optimiser instead of a grid.

### Exercise 8: `recursive_forecast`

In [16]:
def recursive_forecast(model, history, n_lags: int, horizon: int) -> np.ndarray:
    values = list(np.asarray(history, dtype=float))
    if len(values) < n_lags:
        raise ValueError(f"need at least {n_lags} values of history")
    forecasts = []
    for _ in range(horizon):
        row = np.array(values[-n_lags:][::-1]).reshape(1, -1)      # most recent first
        prediction = float(model.predict(row)[0])
        forecasts.append(prediction)
        values.append(prediction)                                 # feed the forecast back in
    return np.array(forecasts)

In [17]:
check_recursive_forecast(recursive_forecast)

PASS  all 4 tests passed for recursive_forecast


The `[::-1]` reverses the last `n_lags` values so the most recent comes first, matching how the model was trained (`lag_1, lag_2, ...`). Getting this order wrong is a classic silent bug: the model still returns numbers, just poor ones. The check uses a model that only looks at `lag_1` to catch it.

From the second step on, the inputs include the model's own forecasts, so errors **compound**: an overestimate at step 1 makes the inputs for step 2 too high, and so on. The check shows this on a series with a strong trend: the recursive forecast drifts further off as the horizon grows.

The alternative, the **direct** strategy from the lesson, trains a separate model for each horizon, with features that only use values available at forecast time. There's also a middle way: train a single model with the horizon as an extra feature.

### Exercise 9: `item_cosine_similarity`

In [18]:
def item_cosine_similarity(R) -> np.ndarray:
    R = np.asarray(R, dtype=float)
    norms = np.sqrt((R ** 2).sum(axis=0))
    safe = np.where(norms == 0, 1.0, norms)              # avoid dividing by zero for empty items
    similarity = (R.T @ R) / np.outer(safe, safe)
    np.fill_diagonal(similarity, 0.0)
    return similarity

In [19]:
check_item_cosine_similarity(item_cosine_similarity)

PASS  all 4 tests passed for item_cosine_similarity


`R.T @ R` computes the dot product of every pair of columns in one matrix multiplication: entry `(i, j)` is the sum over users of `R[u, i] · R[u, j]`. Dividing by the outer product of the column lengths turns the dot products into cosines.

For **purchase data** (0 or 1), the dot product counts the customers who bought both items, and the cosine is that count divided by `√(buyers of i × buyers of j)`. Dividing by the item sizes stops popular items from looking similar to everything just because many people bought them.

Replacing zero norms by 1 leaves those columns as all zeros, so empty items get similarity 0 instead of `nan`.

For large catalogues the full items × items matrix doesn't fit in memory (100,000 items means 10 billion entries). Real systems use sparse matrices, keep only the top few neighbours of each item, or use approximate nearest-neighbour search. The check compares your result with scikit-learn's `cosine_similarity`.

## Hard

### Exercise 10: `holt_forecast`

In [20]:
def holt_forecast(y, alpha: float, beta: float, horizon: int) -> np.ndarray:
    y = np.asarray(y, dtype=float)
    if len(y) < 2:
        raise ValueError("need at least 2 values")
    if not (0 < alpha <= 1 and 0 < beta <= 1):
        raise ValueError("alpha and beta must be in (0, 1]")
    level, trend = y[0], y[1] - y[0]
    for t in range(1, len(y)):
        new_level = alpha * y[t] + (1 - alpha) * (level + trend)
        trend = beta * (new_level - level) + (1 - beta) * trend
        level = new_level
    return level + trend * np.arange(1, horizon + 1)

In [21]:
check_holt_forecast(holt_forecast)

PASS  all 6 tests passed for holt_forecast


Holt's method (Charles Holt, 1957) keeps two running estimates. The **level** is updated like simple smoothing, except that its "previous value" is the last level **plus** the trend, because that's where the series was expected to be by now. The **trend** is a smoothed version of how much the level just changed.

The order of the updates matters: the new trend uses the change from the **old** level to the **new** one, so you need both at once, hence `new_level` as a separate variable. Updating `level` first and then computing `level - level` would always give a trend change of 0. The check catches this with a series whose trend changes.

Two properties the check confirms:

- On a perfectly straight line, Holt's method continues the line exactly, for any `α` and `β`.
- Unlike simple smoothing, it can forecast values beyond anything in the history, which is what you want for a growing series, and dangerous when growth is about to stop. Long-horizon forecasts from a linear trend are often too extreme; the "damped trend" variant (Everette Gardner and Ed McKenzie, 1985) shrinks the trend towards zero over the horizon and tends to forecast better.

### Exercise 11: `als_factorise`

In [22]:
def als_factorise(users, items, values, n_users: int, n_items: int, n_factors: int, reg: float, n_iter: int, seed: int = 0):
    users, items = np.asarray(users), np.asarray(items)
    values = np.asarray(values, dtype=float)
    rng = np.random.default_rng(seed)
    P = rng.normal(0, 0.1, size=(n_users, n_factors))
    Q = rng.normal(0, 0.1, size=(n_items, n_factors))
    rows_of_user = [np.flatnonzero(users == u) for u in range(n_users)]
    rows_of_item = [np.flatnonzero(items == i) for i in range(n_items)]
    identity = reg * np.eye(n_factors)
    for _ in range(n_iter):
        for u, rows in enumerate(rows_of_user):
            if len(rows) == 0:
                P[u] = 0.0
                continue
            Qu = Q[items[rows]]
            P[u] = np.linalg.solve(Qu.T @ Qu + identity, Qu.T @ values[rows])
        for i, rows in enumerate(rows_of_item):
            if len(rows) == 0:
                Q[i] = 0.0
                continue
            Pi = P[users[rows]]
            Q[i] = np.linalg.solve(Pi.T @ Pi + identity, Pi.T @ values[rows])
    return P, Q

In [23]:
check_als_factorise(als_factorise)

PASS  all 5 tests passed for als_factorise


Each update is a **ridge regression** (Module 12): with the item factors fixed, a user's factors are the weights that best predict that user's ratings from the factors of the items they rated, with a penalty `reg` on large weights. The closed-form ridge solution `(XᵀX + λI)⁻¹ Xᵀy` is solved with `np.linalg.solve`, never by inverting the matrix.

Why alternate? The full problem (find `P` and `Q` together) isn't convex, because predictions multiply two unknowns. But with one of them fixed, it's an ordinary least-squares problem with an exact solution. Alternating between the two exact solutions can only lower the regularised squared error, which the check confirms: the objective never increases from one iteration to the next.

Precomputing each user's and item's rows once, before the loop, avoids searching the arrays again on every iteration.

ALS also parallelises well: all users can be updated at the same time, since each user's update only depends on `Q`. That made it a favourite for very large systems (Apache Spark's recommender uses it). The other common method is **stochastic gradient descent**, which updates both factor vectors a little for one rating at a time, as Simon Funk popularised during the Netflix Prize.

### Exercise 12: `recommend_top_k`

In [24]:
def recommend_top_k(scores, seen, popularity, k: int) -> list:
    scores = np.asarray(scores, dtype=float)
    seen = np.asarray(seen, dtype=bool)
    popularity = np.asarray(popularity, dtype=float)
    item_index = np.arange(scores.shape[1])
    recommendations = []
    for user in range(scores.shape[0]):
        candidates = np.flatnonzero(~seen[user])
        # np.lexsort sorts by the LAST key first: score (descending), then popularity (descending), then index
        order = np.lexsort((item_index[candidates], -popularity[candidates], -scores[user, candidates]))
        recommendations.append(candidates[order[:k]])
    return recommendations

In [25]:
check_recommend_top_k(recommend_top_k)

PASS  all 5 tests passed for recommend_top_k


`np.lexsort` sorts by several keys at once. Its quirk is the order: the **last** key in the tuple is the primary one. Negating the score and the popularity turns "highest first" into the ascending order `lexsort` uses.

Filtering out seen items **before** sorting is simpler and safer than setting their scores to `-inf`: with `-inf`, a user with fewer than `k` unseen items would get seen items padded onto the end of their list.

Tie-breaking matters more than it seems. Item-based scores are often exactly equal (many items are similar to nothing the user bought, so they score 0), and a user with no history has a score of 0 for everything. Without a rule, ties are broken by whatever order the sort happens to leave, which can mean recommending item 0, 1, 2... to every new user. Breaking ties by popularity gives a sensible answer automatically, and by index makes it fully reproducible.

Real systems usually add business rules at this stage: remove out-of-stock items, avoid recommending five nearly identical products, mix in some new items. The scoring model and the ranking rules are separate steps, and the second matters as much as the first.

## What's next

The mini project in `04_mini_project.ipynb` uses these tools on real data: a day-ahead forecast of hourly bike rentals, built up from seasonal naive baselines to gradient boosting with lags, and product recommendations for an online gift shop, evaluated on what customers bought next.